# 13장 LangGraph

이 장은 PDF 교재의 LangChain 생태계, Agent, 도구 호출, 메모리, RAG, DB 챗봇 흐름을 바탕으로 `LangGraph`를 사용해 LLM 애플리케이션의 실행 흐름을 그래프 형태로 구성하는 방법을 정리합니다.

앞 장까지는 `chain`, `retriever`, `prompt`, `LLM`, `parser`, `Gradio`를 연결하는 방식으로 실습했습니다. LangGraph는 이 흐름을 더 명확하게 나누어 `상태(State)`, `노드(Node)`, `엣지(Edge)`, `조건 분기(Conditional Edge)`로 표현합니다.

## 이 장에서 다루는 내용

1. LangGraph가 필요한 이유
2. Chain, Agent, Graph의 차이
3. LangGraph 핵심 구성 요소
4. 설치와 기본 import
5. State 정의
6. Node 함수 만들기
7. Edge로 실행 순서 연결하기
8. 조건 분기 그래프 만들기
9. LLM 응답 노드 연결하기
10. RAG 흐름을 LangGraph로 표현하기
11. DB 질의 흐름을 LangGraph로 표현하기
12. 메모리와 반복 실행 구조
13. Gradio와 LangGraph 연결 방향
14. 오류 해결과 설계 팁

이 장은 `LLM/llm.ipynb`의 LangChain, RAG, Gradio 흐름과 `LLM/llm2.ipynb`의 DB 접속 챗봇 흐름을 그래프 구조로 재구성하는 확장 학습입니다.


## 13.1 LangGraph가 필요한 이유

일반적인 LLM 실습은 다음처럼 순차적으로 연결됩니다.

```text
사용자 질문 -> 프롬프트 구성 -> LLM 호출 -> 결과 파싱 -> 출력
```

RAG는 다음처럼 조금 더 길어집니다.

```text
사용자 질문 -> 문서 검색 -> 검색 결과 정리 -> 프롬프트 구성 -> LLM 호출 -> 답변 출력
```

DB 챗봇은 다음처럼 분기와 검증이 필요합니다.

```text
사용자 질문 -> 스키마 확인 -> SQL 생성 -> SQL 검증 -> SQL 실행 -> 결과 해석 -> 답변 출력
```

LangGraph는 이런 흐름을 하나의 긴 함수로 작성하지 않고, 여러 단계로 나눈 뒤 그래프로 연결합니다. 그래서 복잡한 LLM 애플리케이션에서 다음 장점이 있습니다.

- 각 단계를 독립된 함수로 관리할 수 있습니다.
- 실패한 단계만 따로 확인하기 쉽습니다.
- 조건에 따라 다른 경로로 실행할 수 있습니다.
- Agent처럼 반복 판단이 필요한 구조를 표현하기 좋습니다.
- RAG, DB, 도구 호출, 메모리를 하나의 실행 흐름 안에 넣기 쉽습니다.


## 13.2 Chain, Agent, Graph의 차이

| 구분 | Chain | Agent | LangGraph |
|---|---|---|---|
| 핵심 개념 | 정해진 순서로 실행 | LLM이 다음 행동을 선택 | 상태를 중심으로 노드와 엣지를 실행 |
| 실행 흐름 | 대부분 고정 | 동적 판단 | 고정 흐름과 동적 분기를 모두 표현 |
| 적합한 작업 | 단순 Q/A, 요약, 번역 | 도구 선택, 반복 추론 | 복잡한 챗봇, RAG, DB, 다단계 Agent |
| 디버깅 | 중간 단계가 길어지면 어려움 | LLM 판단 추적 필요 | 노드 단위로 확인 가능 |
| 상태 관리 | 별도 메모리 필요 | Agent 내부에 의존 | State 객체에 누적 관리 |

PDF 교재와 노트북에서 이미 다룬 `LangChain`은 LLM 앱의 기본 부품을 연결하는 프레임워크입니다. LangGraph는 LangChain 위에서 복잡한 실행 흐름을 그래프 구조로 안정적으로 관리하도록 도와주는 도구라고 이해하면 됩니다.


In [ ]:
# 교재 위치: 13장 LangGraph - 설치 준비
# 이 셀은 LangGraph 실습에 필요한 패키지를 설치합니다.
# 이미 설치되어 있다면 실행하지 않아도 됩니다.

# langgraph는 상태 기반 그래프 실행을 제공하는 라이브러리입니다.
!pip install langgraph

# langchain은 프롬프트, 모델, 파서, 체인 구성에 사용하는 기본 라이브러리입니다.
!pip install langchain

# langchain-ollama는 Ollama 로컬 LLM을 LangChain 방식으로 호출할 때 사용합니다.
!pip install langchain-ollama

# gradio는 완성한 그래프 기반 챗봇을 웹 UI로 실행할 때 사용합니다.
!pip install gradio


## 13.3 LangGraph 핵심 구성 요소

LangGraph를 사용할 때 가장 중요한 개념은 네 가지입니다.

| 구성 요소 | 의미 |
|---|---|
| State | 그래프 전체에서 공유되는 데이터 구조 |
| Node | 하나의 작업 단계를 수행하는 함수 |
| Edge | 다음에 실행할 노드를 연결하는 선 |
| Conditional Edge | 조건에 따라 다음 노드를 선택하는 분기 |

예를 들어 RAG 챗봇을 LangGraph로 만들면 다음처럼 표현할 수 있습니다.

```text
START
  -> 질문 분석 노드
  -> 문서 검색 노드
  -> 답변 생성 노드
  -> END
```

DB 챗봇은 다음처럼 표현할 수 있습니다.

```text
START
  -> 질문 분석 노드
  -> SQL 생성 노드
  -> SQL 검증 노드
  -> 실행 가능하면 SQL 실행 노드
  -> 실행 불가능하면 오류 안내 노드
  -> END
```


In [ ]:
# 교재 위치: 13장 LangGraph - 기본 import
# 이 셀은 LangGraph 실습에 필요한 기본 모듈을 불러옵니다.

# TypedDict는 딕셔너리 형태의 State가 어떤 키를 가져야 하는지 타입으로 표현합니다.
from typing import TypedDict

# Optional은 값이 있을 수도 있고 None일 수도 있음을 표현합니다.
from typing import Optional

# List는 리스트 타입을 명시할 때 사용합니다.
from typing import List

# Dict는 딕셔너리 타입을 명시할 때 사용합니다.
from typing import Dict

# Any는 값의 타입이 고정되어 있지 않을 때 사용합니다.
from typing import Any

# StateGraph는 LangGraph에서 상태 기반 그래프를 만드는 핵심 클래스입니다.
from langgraph.graph import StateGraph

# START는 그래프의 시작 지점을 의미합니다.
from langgraph.graph import START

# END는 그래프의 종료 지점을 의미합니다.
from langgraph.graph import END


## 13.4 State 정의

State는 그래프가 실행되는 동안 각 노드가 함께 사용하는 데이터입니다. 일반 함수에서는 값을 매개변수와 반환값으로만 넘기지만, LangGraph에서는 State를 중심으로 데이터가 흐릅니다.

아래 예제의 State는 다음 정보를 담습니다.

- `question`: 사용자의 원본 질문
- `route`: 질문이 어떤 유형인지 분류한 결과
- `context`: RAG 검색 결과나 DB 조회 결과처럼 답변에 필요한 근거
- `answer`: 최종 답변
- `error`: 실행 중 발생한 오류 메시지


In [ ]:
# 교재 위치: 13장 LangGraph - State 정의
# 이 셀은 그래프 전체에서 공유할 상태 구조를 정의합니다.

# ChatState 클래스는 LangGraph 노드들이 주고받을 상태의 키와 타입을 정의합니다.
class ChatState(TypedDict):
    # question은 사용자가 입력한 원본 질문을 저장합니다.
    question: str

    # route는 질문을 어떤 처리 경로로 보낼지 저장합니다.
    route: Optional[str]

    # context는 검색 결과, DB 결과, 중간 분석 결과 같은 근거 정보를 저장합니다.
    context: Optional[str]

    # answer는 사용자에게 보여줄 최종 답변을 저장합니다.
    answer: Optional[str]

    # error는 실행 중 문제가 생겼을 때 오류 메시지를 저장합니다.
    error: Optional[str]


## 13.5 Node 함수 만들기

Node는 그래프에서 하나의 작업 단계를 담당하는 함수입니다. Node 함수는 보통 State를 입력으로 받고, 변경할 State 값을 딕셔너리로 반환합니다.

중요한 점은 Node가 전체 State를 모두 다시 만들 필요가 없다는 것입니다. 변경할 값만 반환하면 LangGraph가 기존 State와 합쳐서 다음 노드로 넘깁니다.


In [ ]:
# 교재 위치: 13장 LangGraph - 기본 Node 함수
# 이 셀은 질문 분류, 검색, 답변 생성을 담당하는 예시 노드를 만듭니다.

# route_question 함수는 질문 내용을 보고 처리 경로를 결정하는 노드입니다.
def route_question(state: ChatState) -> Dict[str, Any]:
    # state에서 사용자의 질문을 꺼냅니다.
    question = state["question"]

    # 질문에 DB나 SQL 관련 단어가 있으면 DB 경로로 분류합니다.
    if "DB" in question or "SQL" in question or "데이터베이스" in question:
        # route 값을 db로 설정해 다음 분기에서 DB 처리 노드로 이동하게 합니다.
        return {"route": "db"}

    # 질문에 문서, PDF, 검색 관련 단어가 있으면 RAG 경로로 분류합니다.
    if "문서" in question or "PDF" in question or "검색" in question:
        # route 값을 rag로 설정해 다음 분기에서 RAG 처리 노드로 이동하게 합니다.
        return {"route": "rag"}

    # 특별한 키워드가 없으면 일반 LLM 답변 경로로 분류합니다.
    return {"route": "general"}


# retrieve_context 함수는 RAG 검색을 흉내 내는 노드입니다.
def retrieve_context(state: ChatState) -> Dict[str, Any]:
    # 실제 프로젝트에서는 이 위치에서 retriever.invoke(question)을 호출합니다.
    context = "검색된 문서 내용 예시: RAG는 질문과 관련된 문서를 검색한 뒤 LLM 답변에 근거로 제공합니다."

    # context 키에 검색 결과를 저장합니다.
    return {"context": context}


# query_database 함수는 DB 조회를 흉내 내는 노드입니다.
def query_database(state: ChatState) -> Dict[str, Any]:
    # 실제 프로젝트에서는 이 위치에서 SQL 생성, 검증, 실행을 수행합니다.
    context = "DB 조회 결과 예시: customer 테이블에서 조건에 맞는 3개의 행을 찾았습니다."

    # context 키에 DB 조회 결과를 저장합니다.
    return {"context": context}


# generate_general_answer 함수는 일반 질문에 대한 답변을 만드는 노드입니다.
def generate_general_answer(state: ChatState) -> Dict[str, Any]:
    # 사용자의 질문을 state에서 가져옵니다.
    question = state["question"]

    # 예제에서는 실제 LLM 대신 간단한 문자열 답변을 만듭니다.
    answer = f"일반 답변 예시입니다. 질문 내용: {question}"

    # answer 키에 최종 답변을 저장합니다.
    return {"answer": answer}


# generate_grounded_answer 함수는 context를 근거로 최종 답변을 만드는 노드입니다.
def generate_grounded_answer(state: ChatState) -> Dict[str, Any]:
    # 사용자의 질문을 state에서 가져옵니다.
    question = state["question"]

    # 앞 노드에서 만든 검색 결과 또는 DB 결과를 가져옵니다.
    context = state.get("context")

    # 예제에서는 실제 LLM 대신 context를 포함한 답변 문자열을 만듭니다.
    answer = f"질문: {question}\n\n근거: {context}\n\n답변: 위 근거를 바탕으로 답변을 생성했습니다."

    # answer 키에 최종 답변을 저장합니다.
    return {"answer": answer}


## 13.6 Edge와 조건 분기

Edge는 노드와 노드를 연결합니다. 단순한 순서라면 `add_edge`를 사용합니다.

질문 유형에 따라 RAG, DB, 일반 답변으로 나누고 싶다면 `add_conditional_edges`를 사용합니다.

아래 예제에서는 다음 흐름을 구성합니다.

```text
START
  -> route_question
  -> route 값에 따라 rag / db / general 중 하나 선택
  -> rag이면 retrieve_context -> generate_grounded_answer
  -> db이면 query_database -> generate_grounded_answer
  -> general이면 generate_general_answer
  -> END
```


In [ ]:
# 교재 위치: 13장 LangGraph - 그래프 구성
# 이 셀은 StateGraph를 만들고 노드와 엣지를 연결합니다.

# select_next_node 함수는 route 값에 따라 다음 노드 이름을 반환합니다.
def select_next_node(state: ChatState) -> str:
    # state에서 route 값을 가져옵니다.
    route = state.get("route")

    # route가 rag이면 문서 검색 노드로 이동합니다.
    if route == "rag":
        # retrieve_context는 RAG 검색 역할을 하는 노드 이름입니다.
        return "retrieve_context"

    # route가 db이면 DB 조회 노드로 이동합니다.
    if route == "db":
        # query_database는 DB 조회 역할을 하는 노드 이름입니다.
        return "query_database"

    # 그 외의 질문은 일반 답변 노드로 이동합니다.
    return "generate_general_answer"


# StateGraph 객체를 생성하고 ChatState를 그래프의 상태 타입으로 지정합니다.
graph_builder = StateGraph(ChatState)

# route_question 함수를 route_question이라는 노드 이름으로 등록합니다.
graph_builder.add_node("route_question", route_question)

# retrieve_context 함수를 retrieve_context라는 노드 이름으로 등록합니다.
graph_builder.add_node("retrieve_context", retrieve_context)

# query_database 함수를 query_database라는 노드 이름으로 등록합니다.
graph_builder.add_node("query_database", query_database)

# generate_general_answer 함수를 일반 답변 노드로 등록합니다.
graph_builder.add_node("generate_general_answer", generate_general_answer)

# generate_grounded_answer 함수를 근거 기반 답변 노드로 등록합니다.
graph_builder.add_node("generate_grounded_answer", generate_grounded_answer)

# START에서 route_question 노드로 이어지는 시작 엣지를 추가합니다.
graph_builder.add_edge(START, "route_question")

# route_question 이후에는 select_next_node 함수의 반환값에 따라 다음 노드를 선택합니다.
graph_builder.add_conditional_edges("route_question", select_next_node)

# RAG 검색이 끝나면 근거 기반 답변 생성 노드로 이동합니다.
graph_builder.add_edge("retrieve_context", "generate_grounded_answer")

# DB 조회가 끝나면 근거 기반 답변 생성 노드로 이동합니다.
graph_builder.add_edge("query_database", "generate_grounded_answer")

# 일반 답변 생성이 끝나면 그래프를 종료합니다.
graph_builder.add_edge("generate_general_answer", END)

# 근거 기반 답변 생성이 끝나면 그래프를 종료합니다.
graph_builder.add_edge("generate_grounded_answer", END)

# 구성한 그래프를 실행 가능한 앱으로 컴파일합니다.
chat_graph = graph_builder.compile()


## 13.7 그래프 실행하기

컴파일된 그래프는 `invoke()`로 실행할 수 있습니다. 입력은 State 형식의 딕셔너리입니다.

실행 결과는 마지막 State입니다. 즉, 중간 노드에서 채운 `route`, `context`, `answer` 같은 값이 함께 들어 있습니다.


In [ ]:
# 교재 위치: 13장 LangGraph - 그래프 실행
# 이 셀은 위에서 만든 그래프를 질문 예시로 실행합니다.

# RAG 경로로 분기될 질문을 입력 State로 만듭니다.
rag_input = {
    # question은 사용자가 입력한 질문입니다.
    "question": "PDF 문서에서 RAG 개념을 검색해서 설명해줘",
    # route는 아직 결정되지 않았으므로 None으로 둡니다.
    "route": None,
    # context는 아직 검색 전이므로 None으로 둡니다.
    "context": None,
    # answer는 아직 생성 전이므로 None으로 둡니다.
    "answer": None,
    # error는 아직 오류가 없으므로 None으로 둡니다.
    "error": None,
}

# chat_graph.invoke는 입력 State를 그래프에 넣고 최종 State를 반환합니다.
rag_result = chat_graph.invoke(rag_input)

# 최종 State에서 answer 값을 꺼내 출력합니다.
print(rag_result["answer"])


In [ ]:
# 교재 위치: 13장 LangGraph - DB 경로 실행
# 이 셀은 DB 질문이 DB 처리 노드로 분기되는지 확인하는 예시입니다.

# DB 경로로 분기될 질문을 입력 State로 만듭니다.
db_input = {
    # question에 DB 키워드가 들어 있으므로 route_question 노드에서 db로 분류됩니다.
    "question": "DB에서 고객별 주문 건수를 SQL로 조회해줘",
    # route는 그래프가 실행되면서 결정합니다.
    "route": None,
    # context는 DB 조회 노드에서 채웁니다.
    "context": None,
    # answer는 답변 생성 노드에서 채웁니다.
    "answer": None,
    # error는 오류가 발생했을 때 사용합니다.
    "error": None,
}

# 그래프를 실행하고 최종 State를 받습니다.
db_result = chat_graph.invoke(db_input)

# 최종 답변을 출력합니다.
print(db_result["answer"])


## 13.8 LLM을 연결한 LangGraph

앞 예제는 구조를 쉽게 이해하기 위해 실제 LLM 대신 문자열을 사용했습니다. 실제 챗봇에서는 `ChatOllama`, `ChatOpenAI`, 또는 다른 모델 객체를 노드 안에서 호출하면 됩니다.

아래 코드는 PDF 교재와 기존 노트북에서 사용한 LangChain 방식처럼 프롬프트와 LLM을 연결하되, 그 실행 위치를 LangGraph 노드 안으로 옮깁니다.


In [ ]:
# 교재 위치: 13장 LangGraph - LLM 노드 예시
# 이 셀은 Ollama 로컬 LLM을 LangGraph 노드 안에서 사용하는 예시입니다.

# ChatPromptTemplate은 시스템 메시지와 사용자 메시지를 포함한 프롬프트 템플릿을 만듭니다.
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser는 LLM 응답 객체를 문자열로 변환합니다.
from langchain_core.output_parsers import StrOutputParser

# ChatOllama는 Ollama에서 실행 중인 로컬 LLM을 LangChain 인터페이스로 호출합니다.
from langchain_ollama import ChatOllama

# llm 변수에는 사용할 Ollama 모델을 지정합니다.
llm = ChatOllama(model="llama3.1")

# answer_prompt는 근거와 질문을 바탕으로 답변을 만들기 위한 프롬프트입니다.
answer_prompt = ChatPromptTemplate.from_messages([
    # system 메시지는 모델의 역할과 답변 원칙을 알려줍니다.
    ("system", "너는 LLM 교재 실습을 도와주는 한국어 튜터야. 주어진 근거를 바탕으로 정확하게 답변해."),
    # human 메시지는 실제 질문과 근거를 템플릿 변수로 전달합니다.
    ("human", "질문: {question}\n\n근거: {context}\n\n답변:"),
])

# answer_chain은 프롬프트, LLM, 문자열 파서를 순서대로 연결한 LangChain 체인입니다.
answer_chain = answer_prompt | llm | StrOutputParser()


# generate_llm_answer 함수는 LangGraph 노드에서 LLM 체인을 호출합니다.
def generate_llm_answer(state: ChatState) -> Dict[str, Any]:
    # state에서 사용자 질문을 가져옵니다.
    question = state["question"]

    # state에서 검색 또는 조회 근거를 가져오고, 없으면 빈 문자열을 사용합니다.
    context = state.get("context") or "추가 근거 없음"

    # answer_chain.invoke에 질문과 근거를 전달해 LLM 답변을 생성합니다.
    answer = answer_chain.invoke({"question": question, "context": context})

    # 생성된 답변을 State의 answer 키에 저장합니다.
    return {"answer": answer}


## 13.9 RAG 흐름을 LangGraph로 표현하기

`LLM/llm.ipynb`에서는 PDF, 텍스트, 웹 문서 등을 로드하고 청크로 나눈 뒤, 임베딩하여 벡터스토어에 저장하고, Retriever로 관련 문서를 검색하는 흐름을 다룹니다.

LangGraph에서는 이 과정을 다음 노드로 나눌 수 있습니다.

| 노드 | 역할 |
|---|---|
| analyze_question | 질문 분석 |
| retrieve_documents | 벡터스토어에서 관련 문서 검색 |
| format_documents | 검색 문서를 프롬프트에 넣기 좋게 정리 |
| generate_answer | LLM으로 답변 생성 |

이렇게 나누면 검색 결과가 비었을 때 다른 경로로 보내거나, 검색 결과가 충분할 때만 답변을 생성하는 구조도 만들 수 있습니다.


In [ ]:
# 교재 위치: 13장 LangGraph - RAG 그래프용 State
# 이 셀은 RAG 그래프에서 사용할 상태 구조를 정의합니다.

# RagState 클래스는 RAG 실행 과정에서 필요한 값을 저장합니다.
class RagState(TypedDict):
    # question은 사용자의 질문입니다.
    question: str

    # documents는 retriever가 찾은 문서 목록입니다.
    documents: Optional[List[str]]

    # context는 문서 목록을 하나의 문자열로 정리한 값입니다.
    context: Optional[str]

    # answer는 최종 LLM 답변입니다.
    answer: Optional[str]

    # error는 검색 실패나 답변 실패 같은 오류를 저장합니다.
    error: Optional[str]


# retrieve_documents 함수는 벡터 검색 노드를 단순화한 예시입니다.
def retrieve_documents(state: RagState) -> Dict[str, Any]:
    # 실제 프로젝트에서는 retriever.invoke(state["question"])을 호출합니다.
    documents = [
        # 첫 번째 검색 문서 예시입니다.
        "RAG는 Retrieval-Augmented Generation의 약자로 검색증강생성을 의미합니다.",
        # 두 번째 검색 문서 예시입니다.
        "벡터스토어는 문서 임베딩을 저장하고 질문과 의미적으로 가까운 문서를 찾습니다.",
    ]

    # documents 키에 검색 결과 목록을 저장합니다.
    return {"documents": documents}


# format_documents 함수는 검색 문서를 하나의 context 문자열로 정리합니다.
def format_documents(state: RagState) -> Dict[str, Any]:
    # state에서 문서 목록을 가져오고, 없으면 빈 리스트를 사용합니다.
    documents = state.get("documents") or []

    # 문서 목록을 줄바꿈으로 연결해 프롬프트에 넣기 쉬운 문자열로 만듭니다.
    context = "\n".join(documents)

    # context 키에 정리된 문서 내용을 저장합니다.
    return {"context": context}


# generate_rag_answer 함수는 정리된 context를 바탕으로 답변을 생성합니다.
def generate_rag_answer(state: RagState) -> Dict[str, Any]:
    # 사용자 질문을 가져옵니다.
    question = state["question"]

    # 정리된 검색 근거를 가져옵니다.
    context = state.get("context") or "검색 결과 없음"

    # 예제에서는 실제 LLM 대신 답변 문자열을 생성합니다.
    answer = f"질문 '{question}'에 대해 다음 근거를 사용했습니다.\n{context}"

    # answer 키에 최종 답변을 저장합니다.
    return {"answer": answer}


In [ ]:
# 교재 위치: 13장 LangGraph - RAG 그래프 구성
# 이 셀은 RAG 전용 그래프를 구성합니다.

# RagState를 사용하는 StateGraph 객체를 생성합니다.
rag_graph_builder = StateGraph(RagState)

# retrieve_documents 노드를 그래프에 등록합니다.
rag_graph_builder.add_node("retrieve_documents", retrieve_documents)

# format_documents 노드를 그래프에 등록합니다.
rag_graph_builder.add_node("format_documents", format_documents)

# generate_rag_answer 노드를 그래프에 등록합니다.
rag_graph_builder.add_node("generate_rag_answer", generate_rag_answer)

# START에서 문서 검색 노드로 연결합니다.
rag_graph_builder.add_edge(START, "retrieve_documents")

# 문서 검색 노드 이후 문서 정리 노드로 연결합니다.
rag_graph_builder.add_edge("retrieve_documents", "format_documents")

# 문서 정리 노드 이후 답변 생성 노드로 연결합니다.
rag_graph_builder.add_edge("format_documents", "generate_rag_answer")

# 답변 생성 노드 이후 그래프를 종료합니다.
rag_graph_builder.add_edge("generate_rag_answer", END)

# RAG 그래프를 실행 가능한 객체로 컴파일합니다.
rag_graph = rag_graph_builder.compile()


## 13.10 DB 질의 흐름을 LangGraph로 표현하기

`LLM/llm2.ipynb`와 12장에서는 자연어 질문을 SQL로 바꾸고, DB에서 실행한 뒤, 조회 결과를 자연어로 설명하는 흐름을 다뤘습니다.

LangGraph로 만들면 다음처럼 단계가 분명해집니다.

| 노드 | 역할 |
|---|---|
| read_schema | DB 스키마 읽기 |
| generate_sql | 질문과 스키마를 보고 SQL 생성 |
| validate_sql | 안전한 SQL인지 확인 |
| execute_sql | SQL 실행 |
| explain_result | 결과를 자연어로 설명 |
| reject_query | 위험하거나 실행 불가능한 SQL 안내 |

DB와 연결되는 LLM 앱에서는 조건 분기가 특히 중요합니다. 잘못 생성된 SQL을 바로 실행하면 데이터 변경이나 보안 문제가 생길 수 있기 때문입니다.


In [ ]:
# 교재 위치: 13장 LangGraph - DB 그래프용 State
# 이 셀은 DB 질의 그래프에서 사용할 상태 구조를 정의합니다.

# DbState 클래스는 자연어 질문부터 SQL 실행 결과까지 필요한 값을 저장합니다.
class DbState(TypedDict):
    # question은 사용자가 입력한 자연어 질문입니다.
    question: str

    # schema는 DB 테이블과 컬럼 정보를 저장합니다.
    schema: Optional[str]

    # sql은 LLM이 생성한 SQL 문장을 저장합니다.
    sql: Optional[str]

    # is_safe는 SQL이 안전한지 여부를 저장합니다.
    is_safe: Optional[bool]

    # result는 SQL 실행 결과를 저장합니다.
    result: Optional[str]

    # answer는 최종 자연어 답변을 저장합니다.
    answer: Optional[str]

    # error는 오류 메시지를 저장합니다.
    error: Optional[str]


# read_schema 함수는 DB 스키마를 읽는 노드 예시입니다.
def read_schema(state: DbState) -> Dict[str, Any]:
    # 실제 프로젝트에서는 information_schema나 SQLAlchemy inspector를 사용합니다.
    schema = "customers(id, name, age), orders(id, customer_id, amount, order_date)"

    # schema 키에 스키마 정보를 저장합니다.
    return {"schema": schema}


# generate_sql 함수는 질문과 스키마를 바탕으로 SQL을 생성하는 노드 예시입니다.
def generate_sql(state: DbState) -> Dict[str, Any]:
    # 사용자 질문을 가져옵니다.
    question = state["question"]

    # 스키마 정보를 가져옵니다.
    schema = state.get("schema")

    # 실제 프로젝트에서는 question과 schema를 프롬프트에 넣어 LLM으로 SQL을 생성합니다.
    sql = "SELECT customer_id, COUNT(*) AS order_count FROM orders GROUP BY customer_id;"

    # sql 키에 생성된 SQL을 저장합니다.
    return {"sql": sql}


# validate_sql 함수는 SQL이 읽기 전용인지 검사하는 노드입니다.
def validate_sql(state: DbState) -> Dict[str, Any]:
    # 생성된 SQL을 가져오고, 없으면 빈 문자열로 처리합니다.
    sql = state.get("sql") or ""

    # SQL을 소문자로 바꿔 금지 키워드를 쉽게 검사합니다.
    lowered_sql = sql.lower()

    # 데이터 변경이나 삭제를 일으킬 수 있는 위험 키워드 목록입니다.
    blocked_keywords = ["insert", "update", "delete", "drop", "alter", "truncate"]

    # 위험 키워드가 SQL에 하나라도 들어 있는지 확인합니다.
    has_blocked_keyword = any(keyword in lowered_sql for keyword in blocked_keywords)

    # SELECT로 시작하고 위험 키워드가 없으면 안전하다고 판단합니다.
    is_safe = lowered_sql.strip().startswith("select") and not has_blocked_keyword

    # is_safe 키에 검사 결과를 저장합니다.
    return {"is_safe": is_safe}


# select_db_next_node 함수는 SQL 안전성에 따라 다음 노드를 선택합니다.
def select_db_next_node(state: DbState) -> str:
    # is_safe 값이 True이면 SQL 실행 노드로 이동합니다.
    if state.get("is_safe") is True:
        # execute_sql은 SQL을 실제로 실행하는 노드 이름입니다.
        return "execute_sql"

    # 안전하지 않으면 거절 답변 노드로 이동합니다.
    return "reject_query"


# execute_sql 함수는 SQL 실행 결과를 만드는 노드 예시입니다.
def execute_sql(state: DbState) -> Dict[str, Any]:
    # 실제 프로젝트에서는 pandas.read_sql 또는 SQLAlchemy connection.execute를 사용합니다.
    result = "customer_id=1, order_count=5\ncustomer_id=2, order_count=3"

    # result 키에 SQL 실행 결과를 저장합니다.
    return {"result": result}


# explain_result 함수는 SQL 결과를 자연어로 설명하는 노드입니다.
def explain_result(state: DbState) -> Dict[str, Any]:
    # 실행 결과를 가져옵니다.
    result = state.get("result")

    # 사용자 질문을 가져옵니다.
    question = state["question"]

    # 예제에서는 LLM 대신 문자열로 설명 답변을 만듭니다.
    answer = f"질문 '{question}'에 대한 DB 조회 결과입니다.\n{result}"

    # answer 키에 최종 답변을 저장합니다.
    return {"answer": answer}


# reject_query 함수는 위험한 SQL에 대해 안내 답변을 만드는 노드입니다.
def reject_query(state: DbState) -> Dict[str, Any]:
    # 사용자에게 실행하지 않았다는 안내 메시지를 만듭니다.
    answer = "생성된 SQL이 안전하지 않거나 읽기 전용 SELECT 문이 아니어서 실행하지 않았습니다."

    # answer와 error를 함께 저장합니다.
    return {"answer": answer, "error": "unsafe_sql"}


In [ ]:
# 교재 위치: 13장 LangGraph - DB 그래프 구성
# 이 셀은 DB 질의 전용 그래프를 구성합니다.

# DbState를 사용하는 StateGraph 객체를 생성합니다.
db_graph_builder = StateGraph(DbState)

# read_schema 노드를 등록합니다.
db_graph_builder.add_node("read_schema", read_schema)

# generate_sql 노드를 등록합니다.
db_graph_builder.add_node("generate_sql", generate_sql)

# validate_sql 노드를 등록합니다.
db_graph_builder.add_node("validate_sql", validate_sql)

# execute_sql 노드를 등록합니다.
db_graph_builder.add_node("execute_sql", execute_sql)

# explain_result 노드를 등록합니다.
db_graph_builder.add_node("explain_result", explain_result)

# reject_query 노드를 등록합니다.
db_graph_builder.add_node("reject_query", reject_query)

# START에서 스키마 읽기 노드로 연결합니다.
db_graph_builder.add_edge(START, "read_schema")

# 스키마 읽기 이후 SQL 생성 노드로 연결합니다.
db_graph_builder.add_edge("read_schema", "generate_sql")

# SQL 생성 이후 SQL 검증 노드로 연결합니다.
db_graph_builder.add_edge("generate_sql", "validate_sql")

# SQL 검증 이후 안전성에 따라 execute_sql 또는 reject_query로 분기합니다.
db_graph_builder.add_conditional_edges("validate_sql", select_db_next_node)

# SQL 실행 이후 결과 설명 노드로 연결합니다.
db_graph_builder.add_edge("execute_sql", "explain_result")

# 결과 설명 이후 그래프를 종료합니다.
db_graph_builder.add_edge("explain_result", END)

# 거절 답변 이후 그래프를 종료합니다.
db_graph_builder.add_edge("reject_query", END)

# DB 그래프를 실행 가능한 객체로 컴파일합니다.
db_graph = db_graph_builder.compile()


## 13.11 메모리와 반복 실행 구조

챗봇은 한 번만 답하고 끝나는 경우보다 대화가 이어지는 경우가 많습니다. LangGraph에서는 State에 대화 기록을 넣거나, 체크포인터를 사용해 실행 상태를 저장할 수 있습니다.

기본 아이디어는 다음과 같습니다.

```text
이전 대화 기록 + 새 질문 -> 그래프 실행 -> 새 답변 -> 대화 기록 갱신
```

메모리를 사용할 때는 다음을 주의해야 합니다.

- 대화 기록이 너무 길어지면 토큰 사용량이 증가합니다.
- 중요한 정보만 요약해서 저장하는 방식이 필요할 수 있습니다.
- 개인정보나 민감 정보는 저장하지 않는 설계가 필요합니다.
- RAG 검색 결과와 대화 기록을 구분해서 프롬프트에 넣는 것이 좋습니다.


In [ ]:
# 교재 위치: 13장 LangGraph - 대화 기록 State 예시
# 이 셀은 대화 기록을 State에 포함하는 간단한 예시입니다.

# MemoryState 클래스는 질문, 답변, 대화 기록을 함께 저장합니다.
class MemoryState(TypedDict):
    # question은 현재 사용자의 새 질문입니다.
    question: str

    # history는 이전 대화 기록 목록입니다.
    history: List[str]

    # answer는 현재 질문에 대한 답변입니다.
    answer: Optional[str]


# answer_with_history 함수는 대화 기록을 참고해 답변을 만드는 노드입니다.
def answer_with_history(state: MemoryState) -> Dict[str, Any]:
    # 현재 질문을 가져옵니다.
    question = state["question"]

    # 이전 대화 기록을 가져옵니다.
    history = state.get("history", [])

    # 예제에서는 실제 LLM 대신 대화 기록 개수를 포함한 답변을 만듭니다.
    answer = f"이전 대화 {len(history)}개를 참고했습니다. 현재 질문: {question}"

    # answer 키에 답변을 저장합니다.
    return {"answer": answer}


# update_history 함수는 현재 질문과 답변을 대화 기록에 추가합니다.
def update_history(state: MemoryState) -> Dict[str, Any]:
    # 기존 대화 기록을 복사합니다.
    history = list(state.get("history", []))

    # 현재 질문과 답변을 하나의 문자열로 묶어 기록에 추가합니다.
    history.append(f"Q: {state['question']}\nA: {state.get('answer')}")

    # 갱신된 history를 State에 저장합니다.
    return {"history": history}


## 13.12 Gradio와 LangGraph 연결

앞 장의 Gradio 챗봇에서는 사용자의 입력을 함수에 전달하고, 함수가 답변 문자열을 반환했습니다. LangGraph도 같은 방식으로 연결할 수 있습니다.

핵심은 Gradio 이벤트 함수 안에서 `graph.invoke()`를 호출하는 것입니다.

```text
Gradio 입력 함수
  -> 입력값을 State로 변환
  -> graph.invoke(State)
  -> 결과 State에서 answer 추출
  -> Gradio 출력으로 반환
```


In [ ]:
# 교재 위치: 13장 LangGraph - Gradio 연결 예시
# 이 셀은 LangGraph를 Gradio UI 함수와 연결하는 예시입니다.

# gradio를 gr이라는 별칭으로 불러옵니다.
import gradio as gr


# respond_with_graph 함수는 Gradio에서 받은 질문을 LangGraph로 처리합니다.
def respond_with_graph(message: str, history: List[List[str]]) -> str:
    # Gradio 입력 message를 LangGraph 입력 State로 변환합니다.
    input_state = {
        # question에는 사용자의 현재 메시지를 넣습니다.
        "question": message,
        # route는 그래프 내부에서 결정하므로 None으로 둡니다.
        "route": None,
        # context는 검색 또는 DB 노드에서 채웁니다.
        "context": None,
        # answer는 답변 생성 노드에서 채웁니다.
        "answer": None,
        # error는 오류가 없으므로 None으로 시작합니다.
        "error": None,
    }

    # LangGraph를 실행해 최종 State를 받습니다.
    result_state = chat_graph.invoke(input_state)

    # 최종 State에서 answer를 꺼내 Gradio 출력으로 반환합니다.
    return result_state.get("answer") or "답변을 생성하지 못했습니다."


# ChatInterface는 간단한 챗봇 UI를 빠르게 만들 수 있는 Gradio 컴포넌트입니다.
demo = gr.ChatInterface(
    # fn에는 사용자의 메시지를 처리할 함수를 지정합니다.
    fn=respond_with_graph,
    # title에는 웹 화면 상단에 표시할 제목을 지정합니다.
    title="LangGraph 기반 LLM 챗봇",
    # description에는 챗봇의 간단한 설명을 지정합니다.
    description="질문 유형에 따라 일반 답변, RAG, DB 흐름으로 분기하는 예시입니다.",
)

# launch를 실행하면 로컬 웹 브라우저에서 챗봇 UI를 확인할 수 있습니다.
# demo.launch()


## 13.13 LangGraph 설계 팁

LangGraph를 사용할 때는 처음부터 너무 큰 그래프를 만들기보다, 작은 노드부터 안정적으로 연결하는 것이 좋습니다.

좋은 설계 습관은 다음과 같습니다.

- State에는 꼭 필요한 값만 넣습니다.
- 노드 하나가 너무 많은 일을 하지 않도록 나눕니다.
- LLM 호출, 검색, DB 실행, 후처리를 서로 다른 노드로 분리합니다.
- 조건 분기는 문자열 값을 명확히 정해서 관리합니다.
- DB 실행 전에는 반드시 SQL 검증 노드를 둡니다.
- RAG 검색 결과가 없을 때의 경로를 따로 준비합니다.
- 오류 메시지는 `error` 같은 State 키에 저장해 이후 노드에서 처리합니다.
- Gradio 같은 UI 코드는 그래프 내부가 아니라 바깥쪽 연결 계층에 두는 것이 좋습니다.


## 13.14 자주 발생하는 오류와 해결 방법

| 오류 상황 | 원인 | 해결 방법 |
|---|---|---|
| State 키 오류 | State에 정의하지 않은 키를 사용 | TypedDict와 노드 반환값 확인 |
| 다음 노드 이름 오류 | 조건 분기 함수가 등록되지 않은 노드 이름 반환 | add_node에 등록한 이름과 일치시키기 |
| 그래프 종료 안 됨 | END로 가는 엣지가 없음 | 마지막 노드에서 END 연결 |
| LLM 호출 실패 | Ollama 서버 미실행 또는 모델 없음 | Ollama 실행 후 모델 pull 확인 |
| DB 실행 위험 | 생성 SQL 검증 없음 | validate_sql 노드를 반드시 추가 |
| 답변 근거 누락 | context가 비어 있음 | 검색 결과 없음 분기 또는 안내 답변 추가 |
| UI 응답 없음 | Gradio 함수에서 answer를 반환하지 않음 | graph.invoke 결과에서 answer 추출 |


## 13.15 정리

이 장에서는 LangGraph를 사용해 LLM 애플리케이션의 실행 흐름을 그래프 구조로 만드는 방법을 정리했습니다.

핵심 정리는 다음과 같습니다.

- LangGraph는 복잡한 LLM 앱을 State, Node, Edge로 구성합니다.
- State는 그래프 전체에서 공유되는 실행 데이터입니다.
- Node는 검색, SQL 생성, 답변 생성처럼 하나의 작업 단계를 담당합니다.
- Edge는 노드의 실행 순서를 연결합니다.
- Conditional Edge는 질문 유형이나 검증 결과에 따라 다른 경로로 분기합니다.
- RAG, DB 챗봇, Agent, 메모리 기반 챗봇을 구조적으로 만들 때 유용합니다.
- 앞 장에서 만든 LangChain, RAG, DB 챗봇 코드는 LangGraph 노드로 나누어 확장할 수 있습니다.

다음 장에서는 LangChain 생태계의 관찰과 추적 도구인 LangSmith를 다룹니다. LangSmith는 프롬프트, 체인, LLM 호출, 오류, 실행 시간 등을 확인해 LLM 앱을 디버깅하고 개선하는 데 사용됩니다.
